In [9]:
import pandas as pd

In [10]:
df = pd.read_csv(
    "../datasets/intent_dataset.csv",
    encoding="latin1"
)

In [11]:
print(df.shape)

(1083, 2)


In [12]:
print(df.head())

                                                text       intent
0     I keep having trouble falling asleep at night.  sleep_issue
1  I lie awake for hours before finally drifting ...  sleep_issue
2        Sleep doesn't come easily to me these days.  sleep_issue
3  I struggle to shut my mind off when it's time ...  sleep_issue
4      Most nights, I toss and turn for a long time.  sleep_issue


In [13]:
print(df.columns)


Index(['text', 'intent'], dtype='str')


In [14]:
print(df["intent"].value_counts())

intent
neutral               100
sleep_issue            99
anxiety                99
social_withdrawal      99
hopelessness           99
self_harm              99
Loneliness             98
work_stress            98
general_depression     98
academic_stress        97
relationship_issue     97
Name: count, dtype: int64


In [15]:
print(df.isnull().sum())

text      0
intent    0
dtype: int64


In [16]:
print("Duplicates:", df.duplicated().sum())

Duplicates: 2


In [17]:
df["intent"] = df["intent"].str.strip().str.lower()

In [18]:
df["intent"].value_counts()

intent
neutral               100
sleep_issue            99
anxiety                99
social_withdrawal      99
hopelessness           99
self_harm              99
loneliness             98
work_stress            98
general_depression     98
academic_stress        97
relationship_issue     97
Name: count, dtype: int64

In [19]:
df = df.drop_duplicates()

In [20]:
df.to_csv(
    "../datasets/intent_dataset_clean.csv",
    index=False
)

In [21]:
!pip install transformers datasets torch scikit-learn accelerate


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from datasets import Dataset

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

import torch

c:\Users\SHUBA\OneDrive\Desktop\abhishek_projects\DepressionAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
df = pd.read_csv("../datasets/intent_dataset_clean.csv")

print(df.shape)

df.head()

(1081, 2)


,text,intent
0,I keep having trouble falling asleep at night.,sleep_issue
1,I lie awake for hours before finally drifting ...,sleep_issue
2,Sleep doesn't come easily to me these days.,sleep_issue
3,I struggle to shut my mind off when it's time ...,sleep_issue
4,"Most nights, I toss and turn for a long time.",sleep_issue


In [24]:
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["intent"]
)

print(label_encoder.classes_)

['academic_stress' 'anxiety' 'general_depression' 'hopelessness'
 'loneliness' 'neutral' 'relationship_issue' 'self_harm' 'sleep_issue'
 'social_withdrawal' 'work_stress']


In [25]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [26]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

c:\Users\SHUBA\OneDrive\Desktop\abhishek_projects\DepressionAI\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SHUBA\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [27]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True
)

In [28]:
class IntentDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [29]:
train_dataset = IntentDataset(
    train_encodings,
    list(train_labels)
)

test_dataset = IntentDataset(
    test_encodings,
    list(test_labels)
)

In [30]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_encoder.classes_)
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6120.30it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    acc = accuracy_score(labels, preds)

    f1 = f1_score(
        labels,
        preds,
        average="weighted"
    )

    return {
        "accuracy": acc,
        "f1": f1
    }

In [32]:
training_args = TrainingArguments(

    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    num_train_epochs=5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    learning_rate=2e-5,

    weight_decay=0.01,

    logging_dir="./logs",

    logging_steps=10,

    load_best_model_at_end=True
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [33]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [34]:
trainer.train()

c:\Users\SHUBA\OneDrive\Desktop\abhishek_projects\DepressionAI\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.667964,1.444890,0.797235,0.790205
2,0.689632,0.750780,0.843318,0.843091
3,0.478528,0.540446,0.880184,0.880448
4,0.418716,0.456878,0.870968,0.872942
5,0.269848,0.442123,0.880184,0.881118


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.63it/s]
c:\Users\SHUBA\OneDrive\Desktop\abhishek_projects\DepressionAI\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.87it/s]
c:\Users\SHUBA\OneDrive\Desktop\abhishek_projects\DepressionAI\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.67it/s]
c:\Users\SHUBA\OneDrive\Desktop\abhishek_projects\DepressionAI\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init

TrainOutput(global_step=540, training_loss=0.8544554524951511, metrics={'train_runtime': 176.3155, 'train_samples_per_second': 24.502, 'train_steps_per_second': 3.063, 'total_flos': 22357461369600.0, 'train_loss': 0.8544554524951511, 'epoch': 5.0})

In [35]:
trainer.evaluate()

c:\Users\SHUBA\OneDrive\Desktop\abhishek_projects\DepressionAI\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.269848,0.442123,5,0.880184,0.881118


{'eval_loss': 0.4421226382255554,
 'eval_accuracy': 0.880184331797235,
 'eval_f1': 0.8811176373145813}

In [36]:
model.save_pretrained(
    "../models/intent_model"
)

tokenizer.save_pretrained(
    "../models/intent_model"
)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.29it/s]


('../models/intent_model\\tokenizer_config.json',
 '../models/intent_model\\tokenizer.json')

In [37]:
import pickle

with open(
    "../models/intent_model/label_encoder.pkl",
    "wb"
) as f:

    pickle.dump(
        label_encoder,
        f
    )